<a href="https://colab.research.google.com/github/seonjinj0616-cyber/06-30_CLI_board-teacher-/blob/main/06_30_CLI_%EA%B2%8C%EC%8B%9C%ED%8C%90_%EB%A7%8C%EB%93%A4%EA%B8%B0_(%EA%B0%95%EC%82%AC%EB%8B%98%EC%9A%A9).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
from datetime import datetime

class Article:
  def __init__(self, id, regDate, updateDate, title, body):
    # 생성자 -> 글 하나가 만들어질 때 자동으로 불리는 함수
    # self => 지금 만들어지는 바로 그 글 자신
    self.id = id
    self.regDate = regDate
    self.updateDate = updateDate
    self.title = title
    self.body = body

# ===== 공통 함수 (중복 제거) =====

def now():
    # 현재 시각을 "YYYY-MM-DD HH:MM:SS" 문자열로
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def parse_id(cmd):
    # "article delete 3" -> 3 을 뽑아준다. 잘못된 명령이면 None
    cmdBits = cmd.split(" ")
    if len(cmdBits) < 3:
        print("명령어를 확인해주세요(길이 모자람)")
        return None
    if not cmdBits[2].isdigit():
        print("명령어를 확인해주세요(번호대신 str 입력)")
        return None
    return int(cmdBits[2])

def find_article(articles, article_id):
    # id가 맞는 글을 돌려준다. 없으면 None (found 플래그가 필요 없어짐)
    for article in articles:
        if article.id == article_id:
            return article
    return None

def format_date(regDate):
    # 오늘 쓴 글이면 시:분:초, 이전 글이면 연-월-일 로 보여준다
    today = now().split(" ")[0]        # "2026-06-30"
    datePart = regDate.split(" ")[0]   # 글의 날짜
    timePart = regDate.split(" ")[1]   # 글의 시각
    if datePart == today:
        return timePart
    else:
        return datePart

def makeTestData():
  # 시작하자마자 넣어둘 테스트용 글 3개를 만들어서 리스트로 리턴한다
  # 1번은 이전 날짜 -> 목록 날짜 표기 테스트용
  return [
      Article(1, "2025-12-12 12:12:12","2025-12-12 12:12:12", "제목1", "내용1"),
      Article(2, now(), now(), "제목2", "내용2"),
      Article(3, now(), now(), "제목3", "내용3"),

      # {"id" : 1, "regDate" : "2025-12-12 12:12:12", "updateDate" : "2025-12-12 12:12:12", "title" : "제목1","body":"내용1"},
      # {"id" : 2, "regDate" : now(), "updateDate" : now(), "title" : "제목2","body":"내용2"},
      # {"id" : 3, "regDate" : now(), "updateDate" : now(), "title" : "제목3","body":"내용3"}
          ]

# ===== 메인 =====

print("== CLI 게시판 실행 ==")

last_article_id = 3
articles = makeTestData()

while True:
    cmd = input("명령어 ) ").strip()

    if cmd == "exit":
        break

    elif cmd == "article write":
        last_article_id += 1
        title = input("제목 : ")
        body = input("내용 : ")
        # article = {   -> 코드 곳곳에 흩어져있다
        #     "id": last_article_id,
        #     "regDate": now(),
        #     "updateDate": now(),
        #     "title": title,
        #     "body": body,
        # }
        article = Article(last_article_id, now(), now(), title, body)
        articles.append(article)
        print("{}번 글이 생성되었습니다".format(last_article_id))

    elif cmd == "article list":
        if not articles:
            print("게시글이 없습니다")
        else:
            print("=========================================")
            print("   번호      /     제목       /     내용   /     작성일   ")
            for a in reversed(articles):
                print("   {}      /     {}       /     {}          /     {}   ".format(
                    a.id, a.title, a.body, format_date(a.regDate)))
            print("=========================================")

    elif cmd.startswith("article delete"):
        deletedId = parse_id(cmd)
        if deletedId is None:
            continue

        article = find_article(articles, deletedId)
        if article is None:
            print("{}번 글은 없습니다".format(deletedId))
            continue

        articles.remove(article)
        print("{}번 글이 삭제되었습니다".format(deletedId))

    elif cmd.startswith("article edit"):
        editedId = parse_id(cmd)
        if editedId is None:
            continue

        article = find_article(articles, editedId)
        # 1. 함수가 자기 재료를 눈에 보이게 받는게 더 안전하고 명확함
        # 2. 클래스 사용시 편함(전역에 의존하면 안됨)
        # -> '이 함수가 뭘 필요로 하는지 인자로 다 받는다' 의 습관
        # -> "함수는 독립적이어야 한다"
        if article is None:
            print("{}번 글은 없습니다".format(editedId))
            continue

        print("기존 제목 : {}".format(article.title))
        print("기존 내용 : {}".format(article.body))
        article.title = input("새 제목 : ")
        article.body = input("새 내용 : ")
        article.updateDate = now()
        print("{}번 글이 수정되었습니다".format(editedId))

    elif cmd.startswith("article detail"):
        detailId = parse_id(cmd)
        if detailId is None:
            continue

        article = find_article(articles, detailId)
        if article is None:
            print("{}번 글은 없습니다".format(detailId))
            continue

        print("번호 : {}".format(article.id))
        print("작성 날짜 : {}".format(article.regDate))
        print("수정 날짜 : {}".format(article.updateDate))
        print("제목 : {}".format(article.title))
        print("내용 : {}".format(article.body))

    else:
        print("지원하지 않는 기능입니다")

print("== CLI 게시판 종료 ==")

== CLI 게시판 실행 ==
제목 : asdg
내용 : 456
4번 글이 생성되었습니다
명령어 ) article list
   번호      /     제목       /     내용   /     작성일   
   4      /     asdg       /     456          /     03:14:01   
   3      /     제목3       /     내용3          /     03:13:58   
   2      /     제목2       /     내용2          /     03:13:58   
   1      /     제목1       /     내용1          /     2025-12-12   
명령어 ) exit
== CLI 게시판 종료 ==


In [20]:
from datetime import datetime
from dataclasses import dataclass

@dataclass
class Article:
  id: int
  regDate: str
  updateDate: str
  title: str
  body: str

# ===== 공통 함수 (중복 제거) =====

def now():
    # 현재 시각을 "YYYY-MM-DD HH:MM:SS" 문자열로
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def parse_id(cmd):
    # "article delete 3" -> 3 을 뽑아준다. 잘못된 명령이면 None
    cmdBits = cmd.split(" ")
    if len(cmdBits) < 3:
        print("명령어를 확인해주세요(길이 모자람)")
        return None
    if not cmdBits[2].isdigit():
        print("명령어를 확인해주세요(번호대신 str 입력)")
        return None
    return int(cmdBits[2])

def find_article(articles, article_id):
    # id가 맞는 글을 돌려준다. 없으면 None (found 플래그가 필요 없어짐)
    for article in articles:
        if article.id == article_id:
            return article
    return None

def format_date(regDate):
    # 오늘 쓴 글이면 시:분:초, 이전 글이면 연-월-일 로 보여준다
    today = now().split(" ")[0]        # "2026-06-30"
    datePart = regDate.split(" ")[0]   # 글의 날짜
    timePart = regDate.split(" ")[1]   # 글의 시각
    if datePart == today:
        return timePart
    else:
        return datePart

def makeTestData():
  # 시작하자마자 넣어둘 테스트용 글 3개를 만들어서 리스트로 리턴한다
  # 1번은 이전 날짜 -> 목록 날짜 표기 테스트용
  return [
      Article(1, "2025-12-12 12:12:12","2025-12-12 12:12:12", "제목1", "내용1"),
      Article(2, now(), now(), "제목2", "내용2"),
      Article(3, now(), now(), "제목3", "내용3"),

      # {"id" : 1, "regDate" : "2025-12-12 12:12:12", "updateDate" : "2025-12-12 12:12:12", "title" : "제목1","body":"내용1"},
      # {"id" : 2, "regDate" : now(), "updateDate" : now(), "title" : "제목2","body":"내용2"},
      # {"id" : 3, "regDate" : now(), "updateDate" : now(), "title" : "제목3","body":"내용3"}
          ]

# ===== 메인 =====

print("== CLI 게시판 실행 ==")

last_article_id = 3
articles = makeTestData()

while True:
    cmd = input("명령어 ) ").strip()

    if cmd == "exit":
        break

    elif cmd == "article write":
        last_article_id += 1
        title = input("제목 : ")
        body = input("내용 : ")
        # article = {   -> 코드 곳곳에 흩어져있다
        #     "id": last_article_id,
        #     "regDate": now(),
        #     "updateDate": now(),
        #     "title": title,
        #     "body": body,
        # }
        article = Article(last_article_id, now(), now(), title, body)
        articles.append(article)
        print("{}번 글이 생성되었습니다".format(last_article_id))

    elif cmd == "article list":
        if not articles:
            print("게시글이 없습니다")
        else:
            print("=========================================")
            print("   번호      /     제목       /     내용   /     작성일   ")
            for a in reversed(articles):
                print("   {}      /     {}       /     {}          /     {}   ".format(
                    a.id, a.title, a.body, format_date(a.regDate)))
            print("=========================================")

    elif cmd.startswith("article delete"):
        deletedId = parse_id(cmd)
        if deletedId is None:
            continue

        article = find_article(articles, deletedId)
        if article is None:
            print("{}번 글은 없습니다".format(deletedId))
            continue

        articles.remove(article)
        print("{}번 글이 삭제되었습니다".format(deletedId))

    elif cmd.startswith("article edit"):
        editedId = parse_id(cmd)
        if editedId is None:
            continue

        article = find_article(articles, editedId)
        # 1. 함수가 자기 재료를 눈에 보이게 받는게 더 안전하고 명확함
        # 2. 클래스 사용시 편함(전역에 의존하면 안됨)
        # -> '이 함수가 뭘 필요로 하는지 인자로 다 받는다' 의 습관
        # -> "함수는 독립적이어야 한다"
        if article is None:
            print("{}번 글은 없습니다".format(editedId))
            continue

        print("기존 제목 : {}".format(article.title))
        print("기존 내용 : {}".format(article.body))
        article.title = input("새 제목 : ")
        article.body = input("새 내용 : ")
        article.updateDate = now()
        print("{}번 글이 수정되었습니다".format(editedId))

    elif cmd.startswith("article detail"):
        detailId = parse_id(cmd)
        if detailId is None:
            continue

        article = find_article(articles, detailId)
        if article is None:
            print("{}번 글은 없습니다".format(detailId))
            continue

        print("번호 : {}".format(article.id))
        print("작성 날짜 : {}".format(article.regDate))
        print("수정 날짜 : {}".format(article.updateDate))
        print("제목 : {}".format(article.title))
        print("내용 : {}".format(article.body))

    else:
        print("지원하지 않는 기능입니다")

print("== CLI 게시판 종료 ==")

제목 : asdfg
내용 : 46546
1번 글이 생성되었습니다
[{'id': 1, 'regDate': '2025-12-12 12:12:12', 'updateDate': '2025-12-12 12:12:12', 'title': '제목1', 'body': '내용1'}, {'id': 2, 'regDate': '2026-07-03 02:52:47', 'updateDate': '2026-07-03 02:52:47', 'title': '제목2', 'body': '내용2'}, {'id': 3, 'regDate': '2026-07-03 02:52:47', 'updateDate': '2026-07-03 02:52:47', 'title': '제목3', 'body': '내용3'}, {'id': 1, 'regDate': '2026-07-03 03:09:35', 'updateDate': '2026-07-03 03:09:35', 'title': 'asfg', 'body': '46554'}, {'id': 1, 'regDate': '2026-07-03 03:10:00', 'updateDate': '2026-07-03 03:10:00', 'title': 'asdg', 'body': '45646'}, {'id': 1, 'regDate': '2026-07-03 03:10:52', 'updateDate': '2026-07-03 03:10:52', 'title': 'asdfg', 'body': '46546'}]
